## Задача 1

Лесник решил провести кластеризацию животных по их расположению в лесу. Он разделил карту на квадраты по километровым отметкам: первый квадрат можно описать 0 ≤ x ≤ 1, 0 ≤ y ≤ 1, второй — 1 ≤ x ≤ 2,0 ≤ y ≤ 1 и так далее.

Для файла А леснику нужно определить два соседних квадрата, в которых суммарно находится больше всего животных. Для файла Б леснику нужно определить три соседних квадрата. Квадраты называются соседними, если у них есть общая граница.

Для каждого файла вычислите два числа: S — количество социальных животных в выбранных соседних квадратах, и K — количество остальных животных в выбранных соседних квадратах. Животное называется социальным, если в радиусе 0.1 вокруг него находится как минимум 14 других животных.

В ответе запишите четыре числа: в первой строке S и K для файла А, во второй строке аналогичные данные для файла Б.

In [7]:
import math
from collections import defaultdict

def read_data(name):
    pts = []
    with open(name) as f:
        for line in f:
            x, y = map(float, line.split())
            pts.append((x, y))
    return pts

def social_animals(pts):
    n = len(pts)
    social = [False] * n
    
    for i in range(n):
        cnt = 0
        x1, y1 = pts[i]
        for j in range(n):
            if i == j: continue
            x2, y2 = pts[j]
            if ((x2 - x1)**2 + (y2 - y1)**2)**0.5 <= 0.1:
                cnt += 1
                if cnt >= 14:
                    social[i] = True
                    break
    return social

def count_cells(pts, social):
    cells = defaultdict(lambda: [0, 0])
    for i, (x, y) in enumerate(pts):
        cell = (int(x), int(y))
        cells[cell][0] += 1
        if social[i]:
            cells[cell][1] += 1
    return cells

def solve_file(name, mode):
    pts = read_data(name)
    social = social_animals(pts)
    cells = count_cells(pts, social)
    
    items = list(cells.items())
    best_total = -1
    best_s = 0
    best_k = 0
    
    if mode == 2:
        for i in range(len(items)):
            (x1, y1), (t1, s1) = items[i]
            for j in range(i + 1, len(items)):
                (x2, y2), (t2, s2) = items[j]
                if abs(x1 - x2) + abs(y1 - y2) == 1:
                    total = t1 + t2
                    if total > best_total:
                        best_total = total
                        best_s = s1 + s2
                        best_k = (t1 - s1) + (t2 - s2)
    
    else:
        for i in range(len(items)):
            (x1, y1), (t1, s1) = items[i]
            for j in range(i + 1, len(items)):
                (x2, y2), (t2, s2) = items[j]
                if abs(x1 - x2) + abs(y1 - y2) != 1:
                    continue
                for k in range(j + 1, len(items)):
                    (x3, y3), (t3, s3) = items[k]
                    if (abs(x1 - x3) + abs(y1 - y3) == 1 or 
                        abs(x2 - x3) + abs(y2 - y3) == 1):
                        total = t1 + t2 + t3
                        if total > best_total:
                            best_total = total
                            best_s = s1 + s2 + s3
                            best_k = (t1 - s1) + (t2 - s2) + (t3 - s3)
    
    return best_s, best_k

s_a, k_a = solve_file("1_A.txt", 2)
s_b, k_b = solve_file("1_B.txt", 3)

print(f"{s_a} {k_a}")
print(f"{s_b} {k_b}")

104 453
2156 158


## Задача 2

Научно﻿-﻿исследовательский институт проводит мониторинг экологического состояния различных регионов. Результаты измерений представляются в виде пары чисел: первое — концентрация загрязняющего вещества в почве, второе — концентрация того же вещества в близлежащем водоёме. Для анализа результатов эта пара рассматривается как координаты точки на плоскости, и строится график с точками, которые соответствуют всем измерениям.

По ошибке данные нескольких исследуемых регионов были записаны в один файл. Известно, что измерения относятся к одному региону, если они образуют компактные группы точек на графике. Каждая группа лежит внутри прямоугольника высотой H и шириной.

Перед проведением основного анализа необходимо очистить данные от случайных выбросов при измерении. Для этого используется метод межквартильного размаха:
* для каждой группы точек вычисляется первый квартиль Q1 (значение, ниже которого находится 25% измерений) и третий квартиль Q3 (значение, выше которого находится 25% измерений) отдельно для рядов значений концентрации загрязняющего вещества в почве и в близлежащем водоёме (координаты X и Y)
* вычисляется межквартильный размах IQR - Q3 - Q1 для каждой кординаты
* точка считается выбросом, если хотя бы одна из её координат Х или Y выходит за пределы диапазона |Q1 - 1.5 * IQR; Q3 + 1.5 * IQR|

Для каждого региона необходимо рассчитать индекс экологической опасности I, который определяется как отношение среднего значения измерений к их количеству после удаления выбросов.

Под средним значением измерений в этом случае понимается среднее евклидово расстояние между всеми парами различных точек (измерений) в регионе.

В файле A хранятся данные о трёх регионах, где H=30, W=35. Каждая строка файла содержит два числа: координаты X (концентрация в почве) и Y (концентрация в водоёме), соответствующие одному измерению. Значения даны в условных единицах. Известно, что количество измерений не превышает 1000.

В файле B хранятся данные о пяти регионах, где H=40, W=32. Известно, что количество измерений не превышает 10000. Структура хранения информации в файле B аналогична файлу А.

Для каждого файла определите:
* общее количество выявленных выбросов
* регион с максимальным индексом экологической опасности

В ответе запишите четыре числа: в первой строке — общее количество выбросов и целую часть произведения I×100000 для файла A, во второй строке — аналогичные данные для файла B.

Расчёт квартилей
* Найдите медиану значений данных. Это второй квартиль Q2
* Найдите медиану значений данных, которые находятся ниже второго квартиля. Это первый квартиль Q1
* Найдите медиану значений данных, которые выше второго квартиля. Это третий квартиль Q3

In [9]:
import math
from collections import defaultdict

def read_data(name):
    pts = []
    with open(name) as f:
        for line in f:
            x, y = map(float, line.split())
            pts.append((x, y))
    return pts

def distance(p1, p2):
    return ((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)**0.5

def groups_by_rectangle(points, H, W):
    points = sorted(points)
    groups = []
    used = [False] * len(points)
    
    for i in range(len(points)):
        if used[i]:
            continue
        
        group = [points[i]]
        used[i] = True
        
        for j in range(i + 1, len(points)):
            if used[j]:
                continue
            
            x_min = min(p[0] for p in group)
            x_max = max(p[0] for p in group)
            y_min = min(p[1] for p in group)
            y_max = max(p[1] for p in group)
            
            new_x_min = min(x_min, points[j][0])
            new_x_max = max(x_max, points[j][0])
            new_y_min = min(y_min, points[j][1])
            new_y_max = max(y_max, points[j][1])
            
            if new_x_max - new_x_min <= W and new_y_max - new_y_min <= H:
                group.append(points[j])
                used[j] = True
        
        groups.append(group)
    
    return groups

def quartile(values):
    values = sorted(values)
    n = len(values)
    
    q2 = values[n//2] if n % 2 else (values[n//2 - 1] + values[n//2]) / 2
    
    left = values[:n//2]
    right = values[n//2 + 1:] if n % 2 else values[n//2:]
    
    q1 = left[len(left)//2] if len(left) % 2 else (left[len(left)//2 - 1] + left[len(left)//2]) / 2
    q3 = right[len(right)//2] if len(right) % 2 else (right[len(right)//2 - 1] + right[len(right)//2]) / 2
    
    return q1, q3

def remove_outliers(group):
    xs = [p[0] for p in group]
    ys = [p[1] for p in group]
    
    q1x, q3x = quartile(xs)
    q1y, q3y = quartile(ys)
    
    iqr_x = q3x - q1x
    iqr_y = q3y - q1y
    
    low_x = q1x - 1.5 * iqr_x
    high_x = q3x + 1.5 * iqr_x
    low_y = q1y - 1.5 * iqr_y
    high_y = q3y + 1.5 * iqr_y
    
    filtered = [p for p in group if low_x <= p[0] <= high_x and low_y <= p[1] <= high_y]
    outliers = len(group) - len(filtered)
    
    return filtered, outliers

def avg_distance(group):
    if len(group) < 2:
        return 0
    total = 0
    cnt = 0
    for i in range(len(group)):
        for j in range(i + 1, len(group)):
            total += distance(group[i], group[j])
            cnt += 1
    return total / cnt

def solve_file(name, H, W):
    pts = read_data(name)
    groups = groups_by_rectangle(pts, H, W)
    
    total_outliers = 0
    max_index = -1
    max_region = -1
    
    for idx, group in enumerate(groups):
        filtered, outliers = remove_outliers(group)
        total_outliers += outliers
        
        if len(filtered) >= 2:
            avg_dist = avg_distance(filtered)
            index = avg_dist / len(filtered)
            if index > max_index:
                max_index = index
                max_region = idx
    
    return total_outliers, int(max_index * 100000)

out_a, prod_a = solve_file("2_A.txt", 30, 35)
out_b, prod_b = solve_file("2_B.txt", 40, 32)

print(f"{out_a} {prod_a}")
print(f"{out_b} {prod_b}")

25 3804
467 189


## Задача 3

Кластеризуйте как в предыдущих домашках. Будем называть центром кластера точку этого кластера, сумма расстояний от которой до всех остальных точек кластера минимальна.

Для каждого файла определите координаты центра каждого кластера, затем вычислите два числа: A - среднее арифметическое абсцисс центров кластеров, и среднее B - арифметическое ординат центров кластеров. В ответе запишите четыре числа: в первой строке сначала целую часть произведения A\*10000 затем целую часть произведения B\*10000 для файла А, во второй строке — аналогичные данные для файла Б.


In [4]:
import math

def read_points(filename):
    points = []
    with open(filename, 'r') as f:
        for line in f:
            if line.strip():
                x, y = map(float, line.replace(',', '.').split())
                points.append((x, y))
    return points

def distance(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def find_center(cluster):
    best = None
    min_sum = float('inf')
    for c in cluster:
        s = sum(distance(c, p) for p in cluster)
        if s < min_sum:
            min_sum = s
            best = c
    return best

def clusterize_A(points):
    clusters = [[], [], []]
    for x, y in points:
        if y > 2.57*x - 20.6 and y > 0.7*x - 13.1:
            clusters[0].append((x, y))
        elif y < 0.7*x - 13.1 and y > -9:
            clusters[1].append((x, y))
        elif y < -9:
            clusters[2].append((x, y))
    return [c for c in clusters if c]

def clusterize_B(points):
    clusters = [[], [], [], [], [], []]
    for x, y in points:
        if y > -1.47*x - 22 and y > -8.67*x - 103:
            clusters[0].append((x, y))
        elif y < -8.67*x - 103 and y > 3.57*x + 29.5:
            clusters[1].append((x, y))
        elif y > 3.57*x + 29.5 and y > 1.18*x + 4.7:
            clusters[2].append((x, y))
        elif y < 1.18*x + 4.7 and y > 0.48*x - 2.5:
            clusters[3].append((x, y))
        elif y < 0.48*x - 2.5 and y > -7:
            clusters[4].append((x, y))
        elif y < -7:
            clusters[5].append((x, y))
    return [c for c in clusters if c]

def solve_file(filename, cluster_func):
    points = read_points(filename)
    clusters = cluster_func(points)
    
    centers = []
    for cluster in clusters:
        center = find_center(cluster)
        centers.append(center)
    
    A = sum(c[0] for c in centers) / len(centers)
    B = sum(c[1] for c in centers) / len(centers)
    
    return int(A * 10000), int(B * 10000)

res_A = solve_file('3_A.txt', clusterize_A)
res_B = solve_file('3_B.txt', clusterize_B)

print(res_A[0], res_A[1])
print(res_B[0], res_B[1])

186730 -36346
-100276 66317


## Задача 4

Исследователь анализирует набор объектов, каждый из которых характеризуется пятью числовыми параметрами. Он знает, что объекты образуют несколько групп (кластеров), которые можно выявить при проекции на плоскость только двух параметров из пяти. Значения в одном из столбцов будут соответствовать координатам по оси абсцисс, а из второго — координатам по оси ординат. Каждый кластер можно заключить в квадратную область заданного размера L, причём эти квадраты между собой не пересекаются. Стороны квадратов параллельны координатным осям. Каждый объект должен принадлежать только одному кластеру.

Евклидово расстояние

В файле A хранятся данные о наборе объектов, образующих три кластера. В каждой строке через пробел записаны пять параметров, характеризующих один объект. Все значения представлены с точностью до двух знаков после запятой. Количество объектов в файле А не превышает 1000.
Ы
В файле Б записаны данные о наборе объектов, образующих шесть кластеров, с аналогичной структурой хранения информации. Количество объектов в файле Б не превышает 10000.

Для каждого файла необходимо определить, какая пара параметров позволяет разделить объекты на кластеры, и найти минимальный размер стороны квадрата L, который может содержать все точки одного кластера при проекции на плоскость найденных параметров. Также определите в каждом кластере расстояние между двумя объектами, расположенными дальше всего друг от друга, и вычислите P — среднее арифметическое таких расстояний для всех кластеров.

В ответе запишите четыре числа: в первой строке — целую часть произведения L×10000, затем целую часть произведения P×10000 для файла А, во второй строке — аналогичные значения для файла Б.

In [6]:
import math

def read_points(filename):
    points = []
    with open(filename, 'r') as f:
        for line in f:
            if line.strip():
                parts = line.strip().split()
                if len(parts) >= 2:
                    x = float(parts[0].replace(',', '.'))
                    y = float(parts[1].replace(',', '.'))
                    points.append((x, y))
                elif len(parts) >= 5:
                    x = float(parts[2].replace(',', '.'))
                    y = float(parts[3].replace(',', '.'))
                    points.append((x, y))
    return points

def read_points_5d(filename, idx1, idx2):
    points = []
    with open(filename, 'r') as f:
        for line in f:
            if line.strip():
                parts = line.strip().split()
                if len(parts) >= 5:
                    x = float(parts[idx1].replace(',', '.'))
                    y = float(parts[idx2].replace(',', '.'))
                    points.append((x, y))
    return points

def distance(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def find_center(cluster):
    best = None
    min_sum = float('inf')
    for c in cluster:
        s = sum(distance(c, p) for p in cluster)
        if s < min_sum:
            min_sum = s
            best = c
    return best

def max_distance_in_cluster(cluster):
    max_d = 0
    for i in range(len(cluster)):
        for j in range(i+1, len(cluster)):
            d = distance(cluster[i], cluster[j])
            if d > max_d:
                max_d = d
    return max_d

def clusterize_A(points):
    clusters = [[], [], []]
    
    for x, y in points:
        if y > -0.45*x - 3 and y > 2*x - 19:
            clusters[0].append((x, y))
        elif y < -0.45*x - 3 and y > -1.67*x + 5:
            clusters[1].append((x, y))
        elif y < 2*x - 19 and y < -1.67*x + 5:
            clusters[2].append((x, y))
    
    return [c for c in clusters if c]

def clusterize_B(points):
    clusters = [[], [], [], [], [], []]
    
    for x, y in points:
        if y > -0.5*x + 15:
            clusters[0].append((x, y))
        elif y > 0.5*x + 27 and x > -7:
            clusters[1].append((x, y))
        elif y < -10 and x < -5:
            clusters[2].append((x, y))
        elif y > -15 and y < 10 and x > -4 and x < 2:
            clusters[3].append((x, y))
        elif y < -15 and x > 0 and x < 5:
            clusters[4].append((x, y))
        elif y < -20 and x > 5:
            clusters[5].append((x, y))
    
    return [c for c in clusters if c]

def get_L(cluster):
    xs = [p[0] for p in cluster]
    ys = [p[1] for p in cluster]
    return max(max(xs) - min(xs), max(ys) - min(ys))

def solve_file_A(filename):
    points = read_points(filename)
    clusters = clusterize_A(points)
    
    centers = []
    max_dists = []
    L_values = []
    
    for cluster in clusters:
        center = find_center(cluster)
        centers.append(center)
        max_dists.append(max_distance_in_cluster(cluster))
        L_values.append(get_L(cluster))
    
    L = max(L_values)
    P = sum(max_dists) / len(max_dists)
    
    return int(L * 10000), int(P * 10000)

def solve_file_B(filename):
    points = read_points_5d(filename, 1, 4)
    clusters = clusterize_B(points)
    
    centers = []
    max_dists = []
    L_values = []
    
    for cluster in clusters:
        center = find_center(cluster)
        centers.append(center)
        max_dists.append(max_distance_in_cluster(cluster))
        L_values.append(get_L(cluster))
    
    L = max(L_values)
    P = sum(max_dists) / len(max_dists)
    
    return int(L * 10000), int(P * 10000)

res_A = solve_file_A('4_A.txt')
res_B = solve_file_B('4_B.txt')

print(res_A[0], res_A[1])
print(res_B[0], res_B[1])

765900 541823
242999 135849


In [ ]:
## Задача 5